<h2> Imports, loading event-log function and cleaning pipeline </h2>

In [1]:
import numpy as np
import pandas as pd
import pm4py
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
import statistics
from collections import Counter
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer
from collections import defaultdict
from sklearn.model_selection import GridSearchCV



In [2]:
# Importing dataset from file path
def import_xes(file_path):
    log = pm4py.read_xes(file_path)
    return pm4py.convert_to_dataframe(log)

# Cleaning dataset: removing unnecessary columns, shifting to resource focus
def clean_dataset(df):
    df_final = df[['case:concept:name', 'concept:name', 'org:resource', 'time:timestamp']]
    df_final = df_final.sort_values(['org:resource', 'time:timestamp'])
    return df_final


# creating 80/20 split based on resources, ensuring a resource is in EITHER the test set OR the train set
def create_split(df_clean, test_size):
    resource_traces = (
        df_clean.sort_values(["org:resource", "time:timestamp"])
               .groupby("org:resource")["concept:name"]
               .apply(list)
    )

    resource_traces = resource_traces[resource_traces.apply(len) > 1]

    resources = resource_traces.index.tolist()

    # create set of train/test resource ids
    train_res, test_res = train_test_split(
        resources,
        test_size=test_size,
        random_state=1
    )

    train_traces = resource_traces.loc[train_res]
    test_traces = resource_traces.loc[test_res]

    return train_traces, test_traces


# prefix extraction on set list of prefix lengths, already implicitly buckets on prefix length
def build_prefix_df(resource_traces, prefix_lengths=[10], sliding_window=False):
    all_rows = []
    for length in prefix_lengths:
        for resource, seq in resource_traces.items():

            if(len(seq) < length + 1):
                continue

            if(sliding_window):
                for i in range(length, len(seq)):
                    prefix = seq[i-length:i]
                    next_act = seq[i]

                    all_rows.append({
                    'resource': resource,
                    'prefix': prefix,
                    'prefix_length': length,
                    'last_activity': prefix[-1],
                    'next_activity': next_act
                    })
            else:
                prefix = seq[-(length+1):-1]
                next_act = seq[-1]

                all_rows.append({
                'resource': resource,
                'prefix': prefix,
                'prefix_length': length,
                'last_activity': prefix[-1],
                'next_activity': next_act
                })
            
    return pd.DataFrame(all_rows)




def process_dataset(df, prefix_lengths):
    df_clean = clean_dataset(df)

    train_split, test_split = create_split(df_clean, 0.2)

    train_df = build_prefix_df(train_split, prefix_lengths, True)
    test_df = build_prefix_df(test_split, prefix_lengths, True)

    return train_df, test_df


<h1> Loading event-logs and transforming</h1>

<h4> Loading datasets </h4>

In [3]:
# df_2013, df_2013_prefixes = process_dataset("datasets/BPI_Challenge_2013_incidents.xes")
df_2013 = import_xes("datasets/BPI_Challenge_2013_incidents.xes")
#train_df_2013, test_df_2013 = process_dataset(df_2013)
GLOBAL_prefix_lengths = [10, 20, 30, 40, 50, 75, 100, 125, 150]
train_df_2013, test_df_2013 = process_dataset(df_2013, GLOBAL_prefix_lengths)

/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/pm4py/utils.py:992: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn("Install the optional requirement `rustxes` to import/export files faster.")
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 7554/7554 [00:02<00:00, 3112.21it/s]


In [4]:
def train_RF(X_train, X_test, y_train, y_test):
    rf = RandomForestClassifier(n_estimators=100, random_state=1)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy is: {accuracy}")
    f1score = f1_score(y_test, y_pred, average='weighted')
    print(f"f1score = {f1score}")

    report = classification_report(y_test, y_pred, output_dict=True)
    print(report)
    return rf

In [5]:
# Fitting one hot encoder on the training data and applying on both train and test sets
def ohe_data(train_df, test_df, feature_cols=['last_activity']):
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    encoder.fit(train_df[feature_cols])

    X_train = encoder.transform(train_df[feature_cols])
    X_test = encoder.transform(test_df[feature_cols])

    print(len(X_train))

    return X_train, X_test, encoder

In [ ]:
# Training random forest on un-bucketed 
print("Global OHE performance:")
X_train, X_test, ohe_encoder = ohe_data(train_df_2013, test_df_2013)
global_ohe_rf = train_RF(X_train, X_test, train_df_2013['next_activity'], test_df_2013['next_activity']) 


Global OHE performance:
273719
273719
59045
Accuracy is: 0.593174697264798
f1score = 0.536184662777818


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'Accepted': {'precision': 0.6634254729288976, 'recall': 0.8189069498263801, 'f1-score': 0.7330120047748823, 'support': 39742.0}, 'Completed': {'precision': 0.24817299028931825, 'recall': 0.23861776879391663, 'f1-score': 0.24330159976445187, 'support': 10389.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 8914.0}, 'accuracy': 0.593174697264798, 'macro avg': {'precision': 0.30386615440607195, 'recall': 0.3525082395400989, 'f1-score': 0.32543786817977804, 'support': 59045.0}, 'weighted avg': {'precision': 0.4902044938818863, 'recall': 0.593174697264798, 'f1-score': 0.536184662777818, 'support': 59045.0}}


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [10]:
param_grid = {
    'n_estimators': [50, 100,200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'bootstrap': [True, False]
}

results = []


for length in GLOBAL_prefix_lengths:
    print(f"Searching for length {length}")
    current_train_df = train_df_2013[train_df_2013['prefix_length'] == length]
    current_test_df = test_df_2013[test_df_2013['prefix_length'] == length]

    current_X_train, current_X_test, current_encoder = ohe_data(current_train_df, current_test_df)

    current_y_train = current_train_df['next_activity']
    current_y_test = current_test_df['next_activity']

    rf = train_RF(current_X_train, current_X_test, current_y_train, current_y_test)

    grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                               cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
    
    grid_search.fit(current_X_train, current_y_train)
    
    best_model = grid_search.best_estimator_
    current_y_pred = best_model.predict(current_X_test)
    
    acc = accuracy_score(current_y_test, current_y_pred)
    f1 = f1_score(current_y_test, current_y_pred, average='weighted')

    
    results.append({
        'length': length,
        'support': len(current_y_test),
        'accuracy': acc,
        'f1score': f1,
        'best_params': grid_search.best_params_
    })
total_samples = sum(r['support'] for r in results)
weighted_acc = sum(r['accuracy'] * r['support'] for r in results) / total_samples
weighted_f1 = sum(r['f1score'] * r['support'] for r in results) / total_samples

print("-" * 30)
print(f"Weighted Average Accuracy: {weighted_acc:.4f}")
print(f"Weighted Average F1score: {weighted_f1:.4f}")

    

Searching for length 10
45155
Accuracy is: 0.5942932728647015
f1score = 0.5305143446715961
{'Accepted': {'precision': 0.6583892617449665, 'recall': 0.8321787077619115, 'f1-score': 0.735152688440642, 'support': 7073.0}, 'Completed': {'precision': 0.24574209245742093, 'recall': 0.23245109321058688, 'f1-score': 0.23891188645771733, 'support': 1738.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 1773.0}, 'accuracy': 0.5942932728647015, 'macro avg': {'precision': 0.3013771180674625, 'recall': 0.3548766003241661, 'f1-score': 0.32468819163278645, 'support': 10584.0}, 'weighted avg': {'precision': 0.48033701861424277, 'recall': 0.5942932728647015, 'f1-score': 0.5305143446715961, 'support': 10584.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre

Searching for length 20
40515
Accuracy is: 0.5912940924198563
f1score = 0.5296534196778077
{'Accepted': {'precision': 0.6576760833439856, 'recall': 0.8258426966292135, 'f1-score': 0.7322279940226286, 'support': 6230.0}, 'Completed': {'precision': 0.24601063829787234, 'recall': 0.23521932612841703, 'f1-score': 0.24049398765030874, 'support': 1573.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 1524.0}, 'accuracy': 0.5912940924198563, 'macro avg': {'precision': 0.3012289072139527, 'recall': 0.3536873409192102, 'f1-score': 0.3242406605576458, 'support': 9327.0}, 'weighted avg': {'precision': 0.48078661233789893, 'recall': 0.5912940924198563, 'f1-score': 0.5296534196778077, 'support': 9327.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre

Searching for length 30
36996
Accuracy is: 0.5879396984924623
f1score = 0.5287353565307644
{'Accepted': {'precision': 0.6567936736161035, 'recall': 0.8187847284459581, 'f1-score': 0.7288973990745173, 'support': 5579.0}, 'Completed': {'precision': 0.2466143977191732, 'recall': 0.23731138545953362, 'f1-score': 0.24187347081440055, 'support': 1458.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 1321.0}, 'accuracy': 0.5879396984924623, 'macro avg': {'precision': 0.3011360237784256, 'recall': 0.3520320379684972, 'f1-score': 0.3235902899629726, 'support': 8358.0}, 'weighted avg': {'precision': 0.48143284242388085, 'recall': 0.5879396984924623, 'f1-score': 0.5287353565307644, 'support': 8358.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


...

Searching for length 125
19398
Accuracy is: 0.5985421923184749
f1score = 0.5498844601583288
{'Accepted': {'precision': 0.6753603294440631, 'recall': 0.8078817733990148, 'f1-score': 0.7357009345794393, 'support': 2436.0}, 'Completed': {'precision': 0.2557427258805513, 'recall': 0.2488822652757079, 'f1-score': 0.25226586102719034, 'support': 671.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 460.0}, 'accuracy': 0.5985421923184749, 'macro avg': {'precision': 0.31036768510820484, 'recall': 0.3522546795582409, 'f1-score': 0.32932226520220986, 'support': 3567.0}, 'weighted avg': {'precision': 0.5093302864007815, 'recall': 0.5985421923184749, 'f1-score': 0.5498844601583288, 'support': 3567.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre

Searching for length 150
16765
Accuracy is: 0.6168091168091168
f1score = 0.5701032199692007
{'Accepted': {'precision': 0.6912020905923345, 'recall': 0.8176197836166924, 'f1-score': 0.7491149398159075, 'support': 1941.0}, 'Completed': {'precision': 0.283203125, 'recall': 0.2761904761904762, 'f1-score': 0.2796528447444552, 'support': 525.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 342.0}, 'accuracy': 0.6168091168091168, 'macro avg': {'precision': 0.32480173853077815, 'recall': 0.36460341993572287, 'f1-score': 0.3429225948534542, 'support': 2808.0}, 'weighted avg': {'precision': 0.5307353627011115, 'recall': 0.6168091168091168, 'f1-score': 0.5701032199692007, 'support': 2808.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre

------------------------------
Weighted Average Accuracy: 0.5932
Weighted Average F1score: 0.5362


<h3> Creating last-state encoding </h3>

<h3> Creating bigrams </h3>

In [ ]:
def create_bigram_features(df_train, df_test):

    # all activities in the train set
    train_activities = set([act for prefix in df_train['prefix'] for act in prefix])
    unique_activities = sorted(list(train_activities))

    # all possible bigram transitions
    all_transitions = [f"{a}->{b}" for a in unique_activities for b in unique_activities]

    #print(f"Found {len(unique_activities)} unique activities.")
    print(f"Creating features for {len(all_transitions)} possible bigrams...")

    def _extract_counts(df):
        bigram_rows = []
        for prefix in df['prefix']:
            counts = defaultdict(int)
            
            for i in range(len(prefix) - 1):
                key = f"{prefix[i]}->{prefix[i+1]}"
                counts[key] += 1
            
            row = [counts[t] for t in all_transitions]
            bigram_rows.append(row)
            
        return pd.DataFrame(bigram_rows, columns=all_transitions, index=df.index)

    X_train_bigram = _extract_counts(df_train)
    X_test_bigram = _extract_counts(df_test)
    
    return X_train_bigram, X_test_bigram

X_train2, X_test2 = create_bigram_features(train_df_2013, test_df_2013)
y_train2 = train_df_2013['next_activity']
y_test2 = test_df_2013['next_activity']

# Non-bucketed random forest on bigram encoded 
train_RF(X_train2, X_test2, y_train2, y_test2)

# Bucketed random forest on bigram encoded
for length in GLOBAL_prefix_lengths:
    print(f"Searching for length {length}")
    current_train_df = train_df_2013[train_df_2013['prefix_length'] == length]
    current_test_df = test_df_2013[test_df_2013['prefix_length'] == length]

    current_X_train, current_X_test = create_bigram_features(current_train_df, current_test_df)

    current_y_train = current_train_df['next_activity']
    current_y_test = current_test_df['next_activity']

    rf = train_RF(current_X_train, current_X_test, current_y_train, current_y_test)

    grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                               cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
    
    grid_search.fit(current_X_train, current_y_train)
    
    best_model = grid_search.best_estimator_
    current_y_pred = best_model.predict(current_X_test)
    
    acc = accuracy_score(current_y_test, current_y_pred)
    f1 = f1_score(current_y_test, current_y_pred, average='weighted')

    
    results.append({
        'length': length,
        'support': len(current_y_test),
        'accuracy': acc,
        'f1_score': f1,
        'best_params': grid_search.best_params_
    })
total_samples = sum(r['support'] for r in results)
weighted_acc = sum(r['accuracy'] * r['support'] for r in results) / total_samples
weighted_f1 = sum(r['f1score'] * r['support'] for r in results) / total_samples

print("-" * 30)
print(f"Weighted Average Accuracy: {weighted_acc:.4f}")
print(f"Weighted Average F1score: {weighted_f1:.4f}")
    





Creating features for 16 possible bigrams...
Accuracy is: 0.6339402150901855
f1score = 0.5694206341550441
{'Accepted': {'precision': 0.6837016841657482, 'recall': 0.8897136530622515, 'f1-score': 0.7732207874566746, 'support': 39742.0}, 'Completed': {'precision': 0.2860810562992848, 'recall': 0.15015882183078255, 'f1-score': 0.1969448301982073, 'support': 10389.0}, 'Queued': {'precision': 0.2730666666666667, 'recall': 0.05743773838905093, 'f1-score': 0.09491148391880619, 'support': 8914.0}, 'accuracy': 0.6339402150901855, 'macro avg': {'precision': 0.4142831357105665, 'recall': 0.36577007109402837, 'f1-score': 0.35502570052456267, 'support': 59045.0}, 'weighted avg': {'precision': 0.5517467133995275, 'recall': 0.6339402150901855, 'f1-score': 0.5694206341550441, 'support': 59045.0}}
Searching for length 10
Creating features for 16 possible bigrams...
Accuracy is: 0.6745086923658352
f1score = 0.5775788189572935
{'Accepted': {'precision': 0.6831202299534146, 'recall': 0.9744097271313445, '

/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Searching for length 20
Creating features for 16 possible bigrams...
Accuracy is: 0.6557306743861906
f1score = 0.5733085541583351
{'Accepted': {'precision': 0.680628272251309, 'recall': 0.9390048154093098, 'f1-score': 0.7892074198988196, 'support': 6230.0}, 'Completed': {'precision': 0.324582338902148, 'recall': 0.08645899554990465, 'f1-score': 0.13654618473895583, 'support': 1573.0}, 'Queued': {'precision': 0.41533546325878595, 'recall': 0.08530183727034121, 'f1-score': 0.14153511159499182, 'support': 1524.0}, 'accuracy': 0.6557306743861906, 'macro avg': {'precision': 0.4735153581374143, 'recall': 0.3702552160765185, 'f1-score': 0.3557629054109224, 'support': 9327.0}, 'weighted avg': {'precision': 0.5772331297550256, 'recall': 0.6557306743861906, 'f1-score': 0.5733085541583351, 'support': 9327.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Searching for length 30
Creating features for 16 possible bigrams...
Accuracy is: 0.6293371620004786
f1score = 0.567546799355999
{'Accepted': {'precision': 0.6796089770067465, 'recall': 0.8847463703172611, 'f1-score': 0.7687276125214141, 'support': 5579.0}, 'Completed': {'precision': 0.2924901185770751, 'recall': 0.1522633744855967, 'f1-score': 0.20027063599458728, 'support': 1458.0}, 'Queued': {'precision': 0.30538922155688625, 'recall': 0.07721423164269493, 'f1-score': 0.12326283987915408, 'support': 1321.0}, 'Unmatched': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 0.0}, 'accuracy': 0.6293371620004786, 'macro avg': {'precision': 0.31937207928517697, 'recall': 0.2785559941113882, 'f1-score': 0.27306527209878884, 'support': 8358.0}, 'weighted avg': {'precision': 0.5529323088397536, 'recall': 0.6293371620004786, 'f1-score': 0.567546799355999, 'support': 8358.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zer

Searching for length 40
Creating features for 16 possible bigrams...
Accuracy is: 0.6173036093418259
f1score = 0.5642358871049707
{'Accepted': {'precision': 0.6833228247162674, 'recall': 0.857566765578635, 'f1-score': 0.7605930344767085, 'support': 5055.0}, 'Completed': {'precision': 0.29917550058892817, 'recall': 0.1906906906906907, 'f1-score': 0.23292067858780377, 'support': 1332.0}, 'Queued': {'precision': 0.1836734693877551, 'recall': 0.05483028720626632, 'f1-score': 0.08445040214477212, 'support': 1149.0}, 'accuracy': 0.6173036093418259, 'macro avg': {'precision': 0.3887239315643169, 'recall': 0.36769591449186395, 'f1-score': 0.3593213717364281, 'support': 7536.0}, 'weighted avg': {'precision': 0.5392435591894525, 'recall': 0.6173036093418259, 'f1-score': 0.5642358871049707, 'support': 7536.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Searching for length 50
Creating features for 16 possible bigrams...
Accuracy is: 0.6052823580913469
f1score = 0.5642346134154846
{'Accepted': {'precision': 0.6887810140237325, 'recall': 0.8285034602076125, 'f1-score': 0.7522089141959553, 'support': 4624.0}, 'Completed': {'precision': 0.26379690949227375, 'recall': 0.19274193548387097, 'f1-score': 0.22273998136067102, 'support': 1240.0}, 'Queued': {'precision': 0.2025974025974026, 'recall': 0.0788675429726997, 'f1-score': 0.11353711790393013, 'support': 989.0}, 'accuracy': 0.6052823580913469, 'macro avg': {'precision': 0.38505844203780293, 'recall': 0.3667043128880611, 'f1-score': 0.36282867115351886, 'support': 6853.0}, 'weighted avg': {'precision': 0.5417190147066964, 'recall': 0.6052823580913469, 'f1-score': 0.5642346134154846, 'support': 6853.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
